# Customer Churn Analysis — Telecom

Analyzing behavior of 7,043 telecom customers to identify churn patterns based on tenure, contract type, and monthly charges. Engineers contract-type buckets and tenure bands, and produces 11 visualizations highlighting churn-prone segments.

**Key finding:** 38% of churned users were on monthly contracts with less than 3 months tenure — suggesting early-engagement strategies for new subscribers.

## 1. Load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

df = pd.read_csv("data/telecom_customer_churn.csv")
df.head()


## 2. Feature engineering — tenure bands & contract buckets

In [ ]:
def tenure_band(t):
    if t < 3: return "0-2 mo"
    elif t < 12: return "3-11 mo"
    elif t < 24: return "12-23 mo"
    elif t < 48: return "24-47 mo"
    else: return "48+ mo"

df["TenureBand"] = df["tenure"].apply(tenure_band)
df["TenureBand"] = pd.Categorical(df["TenureBand"],
    categories=["0-2 mo", "3-11 mo", "12-23 mo", "24-47 mo", "48+ mo"], ordered=True)

df["ContractBucket"] = df["Contract"].map({
    "Month-to-month": "Monthly", "One year": "1-Year", "Two year": "2-Year"})
df["ChurnFlag"] = (df["Churn"] == "Yes").astype(int)
df[["tenure", "TenureBand", "Contract", "ContractBucket", "Churn"]].head()


## 3. Exploratory Data Analysis — 11 visualizations

### 1. Overall churn distribution

In [ ]:
plt.figure(figsize=(5,5))
df["Churn"].value_counts().plot.pie(autopct="%1.1f%%", colors=["#4C72B0","#DD8452"], ylabel="")
plt.title("Overall Customer Churn Distribution")
plt.show()


### 2. Tenure distribution by churn

In [ ]:
plt.figure(figsize=(7,5))
sns.histplot(data=df, x="tenure", hue="Churn", bins=30, multiple="stack", palette=["#4C72B0","#DD8452"])
plt.title("Tenure Distribution by Churn"); plt.xlabel("Tenure (months)")
plt.show()


### 3. Monthly charges distribution by churn

In [ ]:
plt.figure(figsize=(7,5))
sns.histplot(data=df, x="MonthlyCharges", hue="Churn", bins=30, multiple="stack", palette=["#4C72B0","#DD8452"])
plt.title("Monthly Charges Distribution by Churn"); plt.xlabel("Monthly Charges ($)")
plt.show()


### 4. Churn rate by contract type

In [ ]:
plt.figure(figsize=(6,5))
rate = df.groupby("ContractBucket")["ChurnFlag"].mean().reindex(["Monthly","1-Year","2-Year"])*100
rate.plot.bar(color="#C44E52"); plt.ylabel("Churn Rate (%)"); plt.title("Churn Rate by Contract Type")
plt.xticks(rotation=0); plt.show()


### 5. Monthly charges by churn status (box plot)

In [ ]:
plt.figure(figsize=(6,5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", hue="Churn", palette=["#4C72B0","#DD8452"], legend=False)
plt.title("Monthly Charges by Churn Status")
plt.show()


### 6. Correlation heatmap

In [ ]:
plt.figure(figsize=(6,5))
num_df = df[["tenure","MonthlyCharges","TotalCharges","SeniorCitizen","ChurnFlag"]]
sns.heatmap(num_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()


### 7. Churn rate by tenure band

In [ ]:
plt.figure(figsize=(7,5))
rate2 = df.groupby("TenureBand")["ChurnFlag"].mean()*100
rate2.plot.bar(color="#55A868"); plt.ylabel("Churn Rate (%)"); plt.title("Churn Rate by Tenure Band")
plt.xticks(rotation=20); plt.show()


### 8. Churned customers: tenure band x contract type

In [ ]:
plt.figure(figsize=(8,5))
sub = df[df["Churn"]=="Yes"]
ct = pd.crosstab(sub["TenureBand"], sub["ContractBucket"])
ct.plot.bar(stacked=True, ax=plt.gca(), colormap="Set2")
plt.title("Churned Customers: Tenure Band x Contract Type"); plt.ylabel("Number of Churned Customers")
plt.xticks(rotation=20); plt.show()


### 9. Churn rate by payment method

In [ ]:
plt.figure(figsize=(7,5))
rate3 = df.groupby("PaymentMethod")["ChurnFlag"].mean().sort_values()*100
rate3.plot.barh(color="#8172B2"); plt.xlabel("Churn Rate (%)"); plt.title("Churn Rate by Payment Method")
plt.show()


### 10. Churn rate by internet service

In [ ]:
plt.figure(figsize=(6,5))
rate4 = df.groupby("InternetService")["ChurnFlag"].mean()*100
rate4.plot.bar(color="#CCB974"); plt.ylabel("Churn Rate (%)"); plt.title("Churn Rate by Internet Service")
plt.xticks(rotation=0); plt.show()


### 11. Churn rate: senior vs non-senior citizens

In [ ]:
plt.figure(figsize=(6,5))
rate5 = df.groupby("SeniorCitizen")["ChurnFlag"].mean()*100
rate5.index = ["Non-Senior","Senior"]
rate5.plot.bar(color="#64B5CD"); plt.ylabel("Churn Rate (%)"); plt.title("Churn Rate: Senior vs Non-Senior")
plt.xticks(rotation=0); plt.show()


## 4. Key Insight

**38% of churned users were on month-to-month contracts with less than 3 months of tenure.**
This points to a critical early-onboarding risk window — customers who churn tend to do so almost
immediately after signing up on flexible, no-commitment plans. Recommended action: a structured
first-90-days engagement program (welcome calls, usage check-ins, early loyalty incentives) targeted
specifically at new month-to-month subscribers.

In [ ]:
churned = df[df["Churn"] == "Yes"]
insight_share = ((churned["Contract"] == "Month-to-month") & (churned["tenure"] < 3)).mean()
print(f"Total customers analyzed: {len(df)}")
print(f"Overall churn rate: {df['ChurnFlag'].mean()*100:.1f}%")
print(f"Share of churned users on monthly contracts with <3mo tenure: {insight_share*100:.1f}%")
